In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

## Question 9.1

In [ ]:
n = 1000
simulations = 1000

coefs = []
ols_se = []
robust_se = []

for i in range(simulations):

    X = np.random.uniform(1, 10, n)
    error = np.random.normal(0, 0.5 * X, n)

    Y = 4 + 7 * X + error

    X = sm.add_constant(X)

    model = sm.OLS(Y, X).fit()

    coefs.append(model.params[1])
    ols_se.append(model.bse[1])

simulation_sd = np.std(coefs)

average_ols_se = np.mean(ols_se)

print(f'Simulation Standard Deviation: {simulation_sd:.3f}')
print(f'Average OLS Standard Error: {average_ols_se:.3f}')

Simulation Standard Deviation: 0.040
Average OLS Standard Error: 0.037


## Question 9.2

In [21]:
n = 1000
simulations = 1000
rho = 0.9

X = np.random.normal(0, 1, n)

X_const = sm.add_constant(X)

coefs = []
ols_se = []

for i in range(simulations): 

    u = np.random.normal(0, 1, n)
    error = np.zeros(n)

    for t in range(1, n):
        error[t] = rho * error[t-1] + u[t]

    Y = 4 + 7 * X + error

    model = sm.OLS(Y, X_const).fit()

    coefs.append(model.params[1])
    ols_se.append(model.bse[1])

simulation_sd = np.std(coefs)
average_ols_se = np.mean(ols_se)

print(f'Simulation Standard Deviation: {simulation_sd:.3f}')
print(f'Average OLS Standard Error: {average_ols_se:.3f}')

Simulation Standard Deviation: 0.078
Average OLS Standard Error: 0.071


In [24]:
u = np.random.normal(0, 1, n)
error = np.zeros(n)

error[0] = np.random.normal(0, 1)

for t in range(1, n):
    error[t] = rho * error[t-1] + u[t]

Y = 4 + 7 * X + error

data = pd.DataFrame({'X' : X, 'Y' : Y})

bootstrap_coefs = []

for i in range(1000):

    bootstrap_data = data.sample(n = n, replace = True)

    X_boot = sm.add_constant(bootstrap_data['X'])

    model = sm.OLS(bootstrap_data['Y'], X_boot).fit()

    bootstrap_coefs.append(model.params['X'])

bootstrap_sd = np.std(bootstrap_coefs)

print(f'Bootstrap Standard Deviation: {bootstrap_sd:.3f}')

Bootstrap Standard Deviation: 0.064


## Question 11.1

In [8]:
n = 100
time = np.arange(n)

event_time = 50

post = (time >= event_time).astype(int)

error = np.random.normal(0, 2, n)

Y = 12 + 0.4 * time + 0.02 * time**2 + 6 * post + 0.3 * (time - event_time) * post + 0.03 * (time - event_time)**2 * post + error

data = pd.DataFrame({'time' : time, 'post' : post, 'Y' : Y})

data.head()

,time,post,Y
0,0,0,11.355623
1,1,0,13.908998
2,2,0,12.813913
3,3,0,13.167375
4,4,0,12.600140


In [9]:
X1 = data[['time', 'post']]
X1['time_sq'] = X1['time']**2
X1 = sm.add_constant(X1)

model1 = sm.OLS(data['Y'], X1).fit()

model1.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                 2.301e+04
Date:                Tue, 24 Nov 2026   Prob (F-statistic):          5.57e-137
Time:                        22:34:29   Log-Likelihood:                -268.56
No. Observations:                 100   AIC:                             545.1
Df Residuals:                      96   BIC:                             555.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.3291      1.122     17.231      0.000      17.102      21.556
time          -0.4549      0.054     -8.382      0.000      -0.563      -0.347
post          -6.1954      1.449     -4.276      0.000      -9.072      -3.319
time_sq        0.0374      0.000     76.880      0.000       0.036       0.038
==============================================================================
Omnibus:                        2.077   Durbin-Watson:                   0.808
Prob(Omnibus):                  0.354   Jarque-Bera (JB):                1.679
Skew:                           0.159   Prob(JB):                        0.432
Kurtosis:                       2.451   Cond. No.                     1.87e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.87e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [10]:
data['time_sq'] = data['time']**2
data['time_after'] = (data['time'] - event_time) * data['post']
data['time_after_sq'] = (data['time'] - event_time)**2 * data['post']

X2 = data[['time', 'time_sq', 'post', 'time_after', 'time_after_sq']]
X2 = sm.add_constant(X2)

model2 = sm.OLS(data['Y'], X2).fit()

model2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      Y   R-squared:                       0.999
Model:                            OLS   Adj. R-squared:                  0.999
Method:                 Least Squares   F-statistic:                 3.594e+04
Date:                Tue, 24 Nov 2026   Prob (F-statistic):          1.47e-152
Time:                        22:34:31   Log-Likelihood:                -219.71
No. Observations:                 100   AIC:                             451.4
Df Residuals:                      94   BIC:                             467.0
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            12.8880      0.916     14.071      0.000      11.069      14.707
time              0.3632      0.086      4.202      0.000       0.192       0.535
time_sq           0.0205      0.002     12.040      0.000       0.017       0.024
post              6.3541      1.350      4.705      0.000       3.673       9.035
time_after        0.2326      0.125      1.866      0.065      -0.015       0.480
time_after_sq     0.0299      0.002     12.372      0.000       0.025       0.035
==============================================================================
Omnibus:                        0.164   Durbin-Watson:                   2.146
Prob(Omnibus):                  0.921   Jarque-Bera (JB):                0.111
Skew:                          -0.078   Prob(JB):                        0.946
Kurtosis:                       2.955   Cond. No.                     2.75e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.75e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [12]:
simulations = 1000

model1_coefs = []
model2_coefs = []

for i in range(1000):

    error = np.random.normal(0, 2, n)

    Y = 12 + 0.4 * time + 0.02 * time**2 + 6 * post + 0.3 * (time - event_time) * post + 0.03 * (time - event_time)**2 * post + error

    data = pd.DataFrame({'time' : time, 'post' : post, 'Y' : Y})

    X1 = sm.add_constant(data[['time', 'post']])
    X1['time_sq'] = X1['time']**2
    model1 = sm.OLS(Y, X1).fit()

    model1_coefs.append(model1.params['post'])

    data['time_sq'] = time**2
    data['time_after'] = (time - event_time) * post
    data['time_after_sq'] = (time - event_time)**2 * post

    X2 = data[['time', 'time_sq', 'post', 'time_after', 'time_after_sq']]
    X2 = sm.add_constant(X2)

    model2 = sm.OLS(Y, X2).fit()
    model2_coefs.append(model2.params[['post', 'time_after', 'time_after_sq']])

model1_average = np.mean(model1_coefs)
model2_average = pd.DataFrame(model2_coefs).mean()

print(f'Model 1 Average Post: {model1_average:.3f}')
print(model2_average)

Model 1 Average Post: -6.672
post             6.022345
time_after       0.295740
time_after_sq    0.030123
dtype: float64


## Question 11.2

In [3]:
time = np.arange(100)
event_time = 50

groups = ['A', 'B', 'C']

starting_values = {'A' : 12, 'B': 18, 'C' : 24}

data = []

for group in groups:
    for t in time: 
        post = (t >= event_time).astype(int)

        y = (starting_values[group] + 0.5 * t + 7 * post + np.random.normal(0, 1))

        data.append([group, t, post, y])

df = pd.DataFrame(data, columns = ['group', 'time', 'post', 'Y'])

df.head()

,group,time,post,Y
0,A,0,0,13.077926
1,A,1,0,13.191808
2,A,2,0,11.804556
3,A,3,0,13.004710
4,A,4,0,14.008303


In [4]:
X = pd.get_dummies(df[['group']], drop_first = True, dtype = int)

X['time'] = df['time']
X['post'] = df['post']

X = sm.add_constant(X)

Y = df['Y']

model = sm.OLS(Y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.997
Model:                            OLS   Adj. R-squared:                  0.997
Method:                 Least Squares   F-statistic:                 2.482e+04
Date:                Tue, 24 Nov 2026   Prob (F-statistic):               0.00
Time:                        22:20:28   Log-Likelihood:                -422.46
No. Observations:                 300   AIC:                             854.9
Df Residuals:                     295   BIC:                             873.4
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         12.1033      0.151     80.097      0.0

## Question 12

In [6]:
time = np.arange(100)
event_time = 50

data = []

for group in ['Treatment', 'Control']:
    for t in time:

        post = (t >= event_time).astype(int)
        treated = int(group == 'Treatment')

        if group == 'Treatment':
            y = 24 + t + 5 * post + np.random.normal(0, 1)
        
        else:
            y = 12 + 0.5 * t + np.random.normal(0, 1)

        data.append([group, t, treated, post, y])

df = pd.DataFrame(data, columns = ['group', 'time', 'treated', 'post', 'Y'])

df.head()

,group,time,treated,post,Y
0,Treatment,0,1,0,24.705771
1,Treatment,1,1,0,25.248958
2,Treatment,2,1,0,26.654677
3,Treatment,3,1,0,25.876003
4,Treatment,4,1,0,28.040230


In [7]:
means = df.groupby(['group', 'post'])['Y'].mean().unstack()

treatment_change = means.loc['Treatment', 1] - means.loc['Treatment', 0]

control_change = means.loc['Control', 1] - means.loc['Control', 0]

did = treatment_change - control_change

print(f'Treatment Group Change: {treatment_change:.3f}')
print(f'Control Group Change: {control_change:.3f}')
print(f'DiD Estimate: {did:.3f}')

Treatment Group Change: 54.896
Control Group Change: 24.697
DiD Estimate: 30.198
